#### Chuẩn bị dữ liệu — bộ TP.HCM (sẵn sàng train)

Notebook làm sạch + kiểm tra + chia dữ liệu, rồi lưu ra file `hcm_train_ready.parquet` để 2
notebook train dùng chung. Mỗi bước ghi rõ **làm gì** và **ý nghĩa**.

Các bước: (1) nạp dữ liệu · (2) kiểm tra tổng quan & kiểu dữ liệu · (3) làm sạch (thiếu/outlier/trùng)
· (4) feature dẫn xuất (giờ VN) · (5) chia dữ liệu (train/val/calib/test theo tháng) · (6) chống rò rỉ
· (7) lưu file sạch.

**Bước 1 — Nạp toàn bộ dữ liệu**

*Làm gì:* đọc 198 file `.csv.gz` của bảng forecasting, ghép thành 1 bảng.
*Ý nghĩa:* gom toàn bộ dữ liệu 3 khu × 3 tháng về một chỗ để xử lý đồng nhất.

In [1]:
import glob, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width",220); pd.set_option("display.max_columns",80)

BASE = Path("../data/synthetic_data/synthetic_quote_context_sandbox_20260727_024458_utc")
FRAC = 1.0   # 1.0 = toan bo (~6.9M dong). Dat 0.2 de chay thu nhanh.
parts = sorted(glob.glob(str(BASE/"hexes/*/synthetic_intern_forecasting_v1_part*.csv.gz")))
print(f"Nap {len(parts)} part (FRAC={FRAC})...")

def doc_toi_uu(p):
    d = pd.read_csv(p)
    if FRAC < 1: d = d.sample(frac=FRAC, random_state=42)
    for c in d.select_dtypes("float64").columns: d[c] = d[c].astype("float32")
    for c in d.select_dtypes("int64").columns: d[c] = pd.to_numeric(d[c], downcast="integer")
    return d

df = pd.concat([doc_toi_uu(p) for p in parts], ignore_index=True)
print(f"Da nap: {df.shape[0]:,} dong x {df.shape[1]} cot | RAM: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print("(giam RAM bang cach ha float64->float32, int64->int32/16 de tranh crash khi FRAC=1.0)")

Nap 198 part (FRAC=1.0)...
Da nap: 6,897,051 dong x 70 cot | RAM: 12.23 GB
(giam RAM bang cach ha float64->float32, int64->int32/16 de tranh crash khi FRAC=1.0)


**Bước 2 — Kiểm tra tổng quan & kiểu dữ liệu**

*Làm gì:* xem kích thước, kiểu dữ liệu, khoảng thời gian, phân bố split/tháng.
*Ý nghĩa:* nắm cấu trúc dữ liệu trước khi xử lý; phát hiện sớm bất thường (cột sai kiểu, thời gian lệch).

In [2]:
df["target_timestamp"] = pd.to_datetime(df.target_timestamp, utc=True)
print("Thoi gian:", df.target_timestamp.min(), "->", df.target_timestamp.max())
print("Split     :", df.split.value_counts().to_dict())
print("Thang     :", df.evaluation_month.value_counts().sort_index().to_dict())
print("Khu       :", df.pickup_location_name.value_counts().to_dict())
print("Dich vu   :", df.service_name.value_counts().to_dict())
print(f"\nGia   : {df.target_shown_price.min():,.0f} -> {df.target_shown_price.max():,.0f} VND")
print(f"He so : {df.target_shown_multiplier.min():.2f} -> {df.target_shown_multiplier.max():.2f}")

Thoi gian: 2026-01-01 00:05:24+00:00 -> 2026-03-30 23:59:55+00:00
Split     : {'train': 4641799, 'test': 864360, 'validation': 774984, 'calibration': 615908}
Thang     : {'2026-01': 2401094, '2026-02': 2169145, '2026-03': 2326812}
Khu       : {'SC Vivo City': 2347784, 'Crescent Mall': 2327972, 'EcoGreen Sài Gòn': 2221295}
Dich vu   : {'Synthetic Premium Car': 3449777, 'Synthetic Standard Car': 3447274}

Gia   : 32,000 -> 959,000 VND
He so : 0.85 -> 1.80


**Bước 3 — Làm sạch: thiếu dữ liệu, outlier, trùng lặp**

*Làm gì:* kiểm tra % thiếu từng cột; kiểm giá/quãng đường bất thường (≤0); bỏ dòng trùng hoàn toàn.
*Ý nghĩa:* dữ liệu bẩn làm model học sai. Bộ này vốn sạch (0% thiếu trừ thời tiết) nên chủ yếu
là **xác nhận**, không phải sửa nhiều. Thời tiết thiếu đã có cờ `weather_missing` → giữ nguyên (cây tự xử lý NaN).

In [3]:
# 3a. Thieu du lieu
miss = (df.isna().mean()*100)
miss_nz = miss[miss>0].round(2)
print("Cot co thieu:", miss_nz.to_dict() if len(miss_nz) else "khong co (tru weather qua co the)")
print(f"weather_missing=1: {df.weather_missing.mean()*100:.2f}%")

# 3b. Outlier co ban
n0 = len(df)
xau = (df.target_shown_price<=0) | (df.quote_distance<=0) | (df.quote_duration<=0)
print(f"\nDong gia/quang duong/thoi luong <= 0: {xau.sum():,}")
df = df[~xau].reset_index(drop=True)

# 3c. Trung lap hoan toan
dup = df.duplicated().sum()
print(f"Dong trung lap hoan toan: {dup:,}")
if dup: df = df.drop_duplicates().reset_index(drop=True)
print(f"\nSau lam sach: {len(df):,} dong (bo {n0-len(df):,})")

Cot co thieu: khong co (tru weather qua co the)
weather_missing=1: 1.73%

Dong gia/quang duong/thoi luong <= 0: 0
Dong trung lap hoan toan: 0

Sau lam sach: 6,897,051 dong (bo 0)


**Bước 4 — Feature dẫn xuất: giờ Việt Nam**

*Làm gì:* dữ liệu gốc dùng giờ UTC. Thêm `gio_vn`, `thu_vn` (UTC+7).
*Ý nghĩa:* TP.HCM là UTC+7. Giữ giờ UTC sẽ đọc sai (giờ cao điểm lệch 7 tiếng). `gio_vn` cho phân
tích/model đọc đúng nhịp sinh hoạt VN.

In [4]:
df["gio_vn"] = (df.target_hour + 7) % 24
df["thu_vn"] = (df.target_day_of_week + (df.target_hour + 7)//24) % 7
print("gio_vn:", sorted(df.gio_vn.unique())[:5], "... | thu_vn:", sorted(df.thu_vn.unique()))
print("Vd: target_hour=1 (UTC) ->", (1+7)%24, "h VN")

gio_vn: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)] ... | thu_vn: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6)]
Vd: target_hour=1 (UTC) -> 8 h VN


**Bước 5 — Chia dữ liệu (train / validation / calibration / test)**

*Làm gì:* dùng cột `split` đã gán sẵn, **theo từng tháng** (`evaluation_month`).
*Ý nghĩa:* đây là bài toán **dự báo** → chia theo thời gian, không ngẫu nhiên. Mỗi tháng là 1 fold
độc lập: 20 ngày đầu = train, sau đó validation / calibration / test theo thứ tự thời gian.

> ⚠️ **Quan trọng (theo tài liệu dataset):** model phải **giữ trong 1 tháng** — train tháng nào
> đánh giá tháng đó. KHÔNG gộp train nhiều tháng rồi chấm tháng trước (lịch sử giá reset theo tháng
> → gộp sẽ rò rỉ tương lai). Notebook train sẽ **lặp theo từng tháng**.

| Tập | Dùng để |
|---|---|
| train | Huấn luyện model |
| validation | Chỉnh siêu tham số (nếu cần) |
| calibration | Dành cho khoảng dự đoán (Conformal, cấu phần iii) |
| test | Đánh giá cuối, model chưa từng thấy |

In [5]:
print("So dong moi (thang x split):")
display(df.groupby(["evaluation_month","split"]).size().unstack("split")
          .reindex(columns=["train","validation","calibration","test"]))

So dong moi (thang x split):


split,train,validation,calibration,test
evaluation_month,,,,
2026-01,1544286,311944,229504,315360
2026-02,1547985,232248,154280,234632
2026-03,1549528,230792,232124,314368


**Bước 6 — Kiểm tra chống rò rỉ**

*Làm gì:* xác nhận feature quan sát chỉ dùng dữ liệu **≤ mốc cắt** (`observation_cutoff_timestamp`),
và độ trễ thực (`actual_observation_age_minutes`) ≥ độ trễ yêu cầu.
*Ý nghĩa:* đảm bảo model không "nhìn lén" thông tin tương lai — điều kiện bắt buộc của bài toán nowcasting.

In [6]:
df["observation_cutoff_timestamp"] = pd.to_datetime(df.observation_cutoff_timestamp, utc=True)
ok_cut = (df.observation_cutoff_timestamp <= df.target_timestamp).all()
ok_age = (df.actual_observation_age_minutes >= 0).all()
print(f"[RO RI] moc cat <= thoi diem dich ? -> {ok_cut}")
print(f"[RO RI] do tre quan sat >= 0 ?       -> {ok_age}")
print("CAM dung lam feature: target_shown_price/multiplier, target_price_per_km, split, cac id.")

[RO RI] moc cat <= thoi diem dich ? -> True
[RO RI] do tre quan sat >= 0 ?       -> True
CAM dung lam feature: target_shown_price/multiplier, target_price_per_km, split, cac id.


**Bước 7 — Lưu dữ liệu sạch (sẵn sàng train)**

*Làm gì:* lưu bảng đã làm sạch + có `gio_vn/thu_vn` ra `hcm_train_ready.parquet`.
*Ý nghĩa:* 2 notebook train nạp thẳng file này, khỏi lặp lại bước làm sạch → nhất quán, nhanh.

In [7]:
OUTP = Path("../data/hcm_train_ready.parquet")
df.to_parquet(OUTP, index=False)
print(f"Da luu: {OUTP}  ({len(df):,} dong x {df.shape[1]} cot)")
print("=> San sang. Mo train/train_gia.ipynb, train/train_heso.ipynb, train/train_hybrid.ipynb.")

Da luu: ..\data\hcm_train_ready.parquet  (6,897,051 dong x 72 cot)
=> San sang. Mo train/train_gia.ipynb, train/train_heso.ipynb, train/train_hybrid.ipynb.
